# Installation and Imports

In [23]:
from qiskit import QuantumCircuit
from mqt.qmap.na.zoned import ZonedNeutralAtomArchitecture
from mqt.qmap.na.zoned import RoutingAwareCompiler
from res_estimate_utils import remove_access_resources
from collections import defaultdict
from qiskit import transpile


# Implementation

## Define Circuit

In [24]:
with open("folded_cultivation_circ.txt", "r") as file:
    qasm_string = file.read()
qiskit_circuit = QuantumCircuit.from_qasm_str(qasm_string)

qiskit_circuit, meas_moves = remove_access_resources(qiskit_circuit, return_meas_reset_moves=True)

qiskit_counts = defaultdict(int)

for inst in qiskit_circuit.data:
     qiskit_counts[len(inst.qubits)] += 1
print(qiskit_counts)

[0, 2, 4, 6, 8, 14, 16, 18, 20, 22, 28, 30, 32, 34, 36, 42, 44, 46, 48, 50, 56, 58, 60, 62, 64, 128, 129, 130]
defaultdict(<class 'int'>, {1: 68, 2: 144, 3: 8})


In [25]:
print(f"Number of qubits in circuit: {qiskit_circuit.num_qubits}")
depth =  qiskit_circuit.depth()
print(f"Circuit depth: {depth}")

Number of qubits in circuit: 28
Circuit depth: 41


## Compile circuits to native gate set

In [26]:
basis_gates = ['rx', 'rz', 'cz']
transpiled_circuit = transpile(qiskit_circuit, basis_gates=basis_gates)

## Define Architecture

In [27]:
arch = ZonedNeutralAtomArchitecture.from_json_string("""{
  "name": "Architecture with one entanglement and one storage zone",
  "operation_duration": {"rydberg_gate": 0.36, "single_qubit_gate": 52, "atom_transfer": 15},
  "operation_fidelity": {"rydberg_gate": 0.995, "single_qubit_gate": 0.9997, "atom_transfer": 0.999},
  "qubit_spec": {"T": 1.5e6},
  "storage_zones": [{
    "zone_id": 0,
    "slms": [{"id": 0, "site_separation": [3, 3], "r": 20, "c": 100, "location": [0, 0]}],
    "offset": [0, 0],
    "dimension": [297, 57]
  }],
  "entanglement_zones": [{
    "zone_id": 0,
    "slms": [
      {"id": 1, "site_separation": [12, 10], "r": 7, "c": 20, "location": [35, 67]},
      {"id": 2, "site_separation": [12, 10], "r": 7, "c": 20, "location": [37, 67]}
    ],
    "offset": [35, 67],
    "dimension": [230, 60]
  }],
  "aods": [{"id": 0, "site_separation": 2, "r": 100, "c": 100}],
  "rydberg_range": [[[30, 62], [270, 132]]]
}""")

## Compile for movements using mqt

In [28]:
depth =  transpiled_circuit.depth()
print(f"Circuit depth: {depth}")

Circuit depth: 92


In [29]:
from res_estimate_utils import calculate_movements_for_arch

compiler = RoutingAwareCompiler(arch)

compiler_moves = calculate_movements_for_arch(arch, transpiled_circuit , compiler)

In [30]:
from mqt.core import load

circ = load(transpiled_circuit)
code = compiler.compile(circ)

In [31]:
print(f"Compiler predicted move: {compiler_moves}, (reset+meas)moves: {meas_moves}")
print(f"Total: {compiler_moves+meas_moves}(moves)")

Compiler predicted move: 632, (reset+meas)moves: 93
Total: 725(moves)
